# InoveHub
## Sistema de Incubadora de Empresas

In [2]:
%pip install matplotlib seaborn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 1.1 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/8.3 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.3 MB 1.1 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.3 MB 1.1 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.3 MB 1.1 MB/s eta 0:00:06
   -------- ------------------------------- 1.8/8.3 MB 1.1 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.3 MB 1.1 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/8.3 MB 1.1 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/8.3 MB 1.1 MB/s eta 0:00:06
   ------------ --------------------------- 2.6/8.3 MB 1.1 MB/s eta 0:00:05
   ------------- ----------------

In [ ]:
import os
import pandas as pd
import panel as pn
import plotly.express as px
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Inicializa as extensões para o navegador reconhecer os gráficos e tabelas
pn.extension('plotly', 'tabulator', notifications=True)
load_dotenv()

# Conexão com o Banco de Dados [cite: 516]
try:
    engine = create_engine(os.getenv("DATABASE_URL"))
except Exception as e:
    print(f"Erro na conexão: {e}")

# --- FUNÇÕES DO CRUD (6,0 Pontos) [cite: 513] ---

def carregar_dados(busca=""):
    """Consulta com filtragem [cite: 513]"""
    query = "SELECT * FROM Mentor"
    if busca:
        query += f" WHERE nome ILIKE '%%{busca}%%' OR area_especialidade ILIKE '%%{busca}%%'"
    return pd.read_sql(query, engine)

def salvar_mentor(event):
    """Inclusão e Edição [cite: 513, 514]"""
    try:
        with engine.begin() as conn:
            # Lógica de Upsert (Insere ou atualiza se o e-mail já existir) [cite: 158]
            sql = text("""
                INSERT INTO Mentor (nome, email, telefone, area_especialidade, biografia)
                VALUES (:nome, :email, :tel, :area, :bio)
                ON CONFLICT (email) DO UPDATE SET 
                nome = EXCLUDED.nome, 
                telefone = EXCLUDED.telefone, 
                area_especialidade = EXCLUDED.area_especialidade, 
                biografia = EXCLUDED.biografia
            """)
            conn.execute(sql, {
                "nome": nome_in.value, "email": email_in.value, 
                "tel": tel_in.value, "area": area_in.value, "bio": bio_in.value
            })
        pn.state.notifications.success("Mentor salvo com sucesso!")
        atualizar_view()
    except Exception as e:
        pn.state.notifications.error(f"Erro ao salvar: {e}")

def deletar_mentor(event):
    """Remoção [cite: 513, 514]"""
    if not email_in.value:
        pn.state.notifications.warning("Insira o e-mail para identificar o mentor.")
        return
    try:
        with engine.begin() as conn:
            conn.execute(text("DELETE FROM Mentor WHERE email = :email"), {"email": email_in.value})
        pn.state.notifications.danger("Mentor removido.")
        atualizar_view()
    except Exception as e:
        pn.state.notifications.error(f"Erro ao remover: {e}")

# --- GRÁFICO POR AGREGAÇÃO (2,0 Pontos)  ---

def criar_grafico():
    """Query: SELECT area_especialidade, COUNT(*) FROM Mentor GROUP BY... """
    df = pd.read_sql("SELECT area_especialidade, COUNT(*) as total FROM Mentor GROUP BY area_especialidade", engine)
    if df.empty:
        return pn.pane.Markdown("Sem dados")
    
    fig = px.bar(df, y='area_especialidade', x='total', orientation='h', 
                 title="Mentores por Especialidade", color='total', template="plotly_white")
    return fig

# --- INTERFACE ---

busca_input = pn.widgets.TextInput(name="Buscar (Nome ou Área)")
nome_in = pn.widgets.TextInput(name="Nome")
email_in = pn.widgets.TextInput(name="E-mail")
tel_in = pn.widgets.TextInput(name="Telefone")
area_in = pn.widgets.TextInput(name="Área")
bio_in = pn.widgets.TextAreaInput(name="Biografia")

btn_save = pn.widgets.Button(name="Salvar/Editar", button_type="success")
btn_del = pn.widgets.Button(name="Remover", button_type="danger")

tabela = pn.widgets.Tabulator(carregar_dados(), sizing_mode='stretch_width')

def atualizar_view(event=None):
    tabela.value = carregar_dados(busca_input.value)
    grafico_pane.object = criar_grafico()

btn_save.on_click(salvar_mentor)
btn_del.on_click(deletar_mentor)
busca_input.param.watch(atualizar_view, 'value')

grafico_pane = pn.pane.Plotly(criar_grafico())

# --- LAYOUT E EXECUÇÃO NO LOCALHOST ---

template = pn.template.FastListTemplate(
    title='Sistema de Incubadora - Módulo Esdras',
    sidebar=[busca_input],
    main=[
        pn.Row(
            pn.Column("### Cadastro", nome_in, email_in, tel_in, area_in, bio_in, pn.Row(btn_save, btn_del), width=350),
            pn.Column("### Análise", grafico_pane)
        ),
        pn.layout.Divider(),
        pn.Column("### Listagem", tabela)
    ]
)

# ESTE COMANDO ABRE O NAVEGADOR AUTOMATICAMENTE
template.show(port=5006)

BokehModel(combine_events=True, render_bundle={'docs_json': {'be2cd150-01fe-469d-9bc7-c5f9ce6fbac2': {'version…

OSError: [WinError 10048] Normalmente é permitida apenas uma utilização de cada endereço de soquete (protocolo/endereço de rede/porta)

In [5]:
import os
import pandas as pd
import panel as pn
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv

pn.extension('plotly') # Habilita o Plotly no Panel
load_dotenv()

@pn.cache
def load_data():
    DB_URL = os.getenv("DATABASE_URL")
    engine = create_engine(DB_URL)
    
    # Carregando tabelas
    df_emp = pd.read_sql("SELECT * FROM Empresa", engine)
    
    df_inv = pd.read_sql("SELECT * FROM Investimento", engine)
    df_inv['data_aporte'] = pd.to_datetime(df_inv['data_aporte'])
    
    df_cont = pd.read_sql("SELECT * FROM Contato", engine)
    
    return df_emp, df_inv, df_cont

# Carrega os dados uma vez
df_empresa, df_investimento, df_contato = load_data()

# Widget para filtrar Áreas de Atuação (Empresas)
areas_disponiveis = list(df_empresa['area_atuacao'].unique()) if not df_empresa.empty else []
filtro_area = pn.widgets.MultiChoice(
    name='Filtrar por Área de Atuação', 
    options=areas_disponiveis,
    value=areas_disponiveis, # Começa com todas selecionadas
    solid=False
)

# Widget para definir Top N cargos (Contatos)
slider_top_cargos = pn.widgets.IntSlider(
    name='Top N Cargos', start=3, end=20, step=1, value=10
)


def criar_grafico_funil_empresas(areas):
    # Filtra o DataFrame com base no widget
    if not areas: 
        df_filtrado = df_empresa # Se nada selecionado, mostra tudo
    else:
        df_filtrado = df_empresa[df_empresa['area_atuacao'].isin(areas)]
        
    if df_filtrado.empty:
        return pn.pane.Markdown("### Sem dados para exibir")

    # Contagem
    contagem = df_filtrado['status_atual'].value_counts().reset_index()
    contagem.columns = ['Status', 'Quantidade']
    
    # Plotly Bar Chart
    fig = px.bar(contagem, x='Status', y='Quantidade', color='Status',
                 title="Funil de Empresas na Incubadora (Status)", template="plotly_white")
    return fig

def criar_grafico_investimentos(areas_dummy): # Recebe areas só para atualizar junto, se quiser
    if df_investimento.empty: return pn.pane.Markdown("### Sem investimentos")
    
    # Agrupamento
    df_agrupado = df_investimento.groupby(df_investimento['data_aporte'].dt.to_period('M').astype(str))['valor'].sum().reset_index()
    
    fig = px.line(df_agrupado, x='data_aporte', y='valor', markers=True,
                  title="Evolução Financeira", template="plotly_white")
    fig.update_layout(xaxis_title="Mês/Ano", yaxis_title="Valor (R$)")
    return fig

def criar_grafico_contatos(top_n):
    if df_contato.empty: return pn.pane.Markdown("### Sem contatos")
    
    top_cargos = df_contato['cargo'].value_counts().nlargest(top_n).reset_index()
    top_cargos.columns = ['Cargo', 'Quantidade']
    
    fig = px.bar(top_cargos, y='Cargo', x='Quantidade', orientation='h',
                 title=f"Top {top_n} Cargos na Rede", color='Quantidade', template="plotly_white")
    fig.update_layout(yaxis={'categoryorder':'total ascending'}) # Ordena do maior pro menor
    return fig

grafico_empresa_view = pn.bind(criar_grafico_funil_empresas, areas=filtro_area)
grafico_invest_view = pn.bind(criar_grafico_investimentos, areas_dummy=filtro_area)
grafico_contato_view = pn.bind(criar_grafico_contatos, top_n=slider_top_cargos)

template = pn.template.FastListTemplate(
    title='InoveHub Dashboard',
    sidebar=[
        pn.pane.Markdown("## Filtros Gerais"),
        filtro_area,
        pn.layout.Divider(),
        pn.pane.Markdown("## Configuração Contatos"),
        slider_top_cargos,
        pn.layout.Divider(),
        pn.pane.Markdown("Dados carregados do PostgreSQL.")
    ],
    main=[
        pn.Row(
            pn.Column(grafico_empresa_view, margin=(10,10)),
            pn.Column(grafico_invest_view, margin=(10,10))
        ),
        pn.Row(
            pn.Column(grafico_contato_view, margin=(10,10))
        )
    ],
    accent_base_color="#2ecc71",
    header_background="#2c3e50"
)

# Comando para mostrar no notebook ou preparar para servir
template.servable();
template.show()

C:\Users\Esdras Levy\AppData\Local\Temp\ipykernel_10084\3797316820.py:8: UserWarning:

Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.



Launching server at http://localhost:63296
